In [1]:
!pip install -q networkx scikit-learn joblib spacy
!python -m spacy download es_core_news_sm -q

!git clone -q https://github.com/osvfj/chatbot.git
import sys
sys.path.insert(0, "/content/chatbot/ml-core/src")
print("Listo: repositorio clonado y dependencias instaladas.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 95.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Listo: repositorio clonado y dependencias instaladas.


In [2]:
import json
from ml_core.core import nlp

data_path = "/content/chatbot/ml-core/src/ml_core/data/intents.json"
records = json.loads(open(data_path, encoding="utf-8").read())
clases = sorted(set(r["intent"] for r in records))

print(f"Total de frases de entrenamiento: {len(records)}")
print(f"Cantidad de intenciones distintas: {len(clases)}")
print("Intenciones:", clases)

print()
print("--- Normalización de modismos dominicanos ---")
frase = "klk manito, tato ese chin de cafe"
print("Original:  ", frase)
print("Normalizada:", nlp.normalize(frase))

Total de frases de entrenamiento: 125
Cantidad de intenciones distintas: 20
Intenciones: ['agradecimiento', 'analizar_foto', 'arañita_roja', 'asistencia_tecnica', 'broca', 'cercospora', 'clima_altitud', 'control_biologico', 'despedida', 'manejo_integrado', 'minador', 'nutricion', 'ojo_gallo', 'phoma', 'poda', 'riego', 'roya', 'saludo', 'sombra_cafetal', 'variedades_resistentes']

--- Normalización de modismos dominicanos ---
Original:   klk manito, tato ese chin de cafe
Normalizada: que tal amigo, esta bien ese un poco de cafe


In [3]:
from ml_core.core.search import KnowledgeGraph

kg = KnowledgeGraph()

consulta = "tengo hojas con manchas amarillas, que puede ser"

for algoritmo in ["bfs", "dfs", "astar"]:
    resultado = kg.search(consulta, algorithm=algoritmo)
    print(f"--- {algoritmo.upper()} ---")
    print("Ruta recorrida:", resultado["path"])
    print("Costo:", resultado["cost"])
    print("Nodo encontrado:", resultado.get("node"))
    print("Respuesta:", resultado["response"])
    print()

--- BFS ---
Ruta recorrida: ['inicio', 'roya']
Costo: 1
Nodo encontrado: roya
Respuesta: # Roya del cafeto

La roya del cafeto es causada por el hongo *Hemileia vastatrix*. En la República Dominicana se considera una de las principales limitantes tecnológicas del cultivo. El síntoma más característico es la aparición de manchas amarillentas en la superficie superior de la hoja y un polvo amarillo-anaranjado formado por esporas en el envés.

## Qué observar

- Revisar especialmente el envés de las hojas, no solo la cara superior.
- Buscar pústulas o polvo anaranjado asociado a las manchas.
- Observar si existe defoliación o pérdida prematura de hojas.
- Comparar hojas nuevas y maduras, porque una sola hoja no representa necesariamente toda la planta.

La roya se favorece por ambientes cálidos, húmedos y lluviosos. La sombra no tiene un efecto único: una sombra adecuada puede reducir el estrés de la planta, mientras una copa densa puede retener humedad y modificar la dispersión de las es

In [4]:
from ml_core.core.rules import RuleEngine

motor = RuleEngine()

print("--- Caso 1: roya en época de lluvias ---")
hechos_1 = {"sintoma": "pustulas_amarillas", "estacion": "lluvias"}
resultado_1 = motor.evaluate(hechos_1)
for regla in resultado_1["applied"]:
    print(f"[{regla['id']}] {regla['explanation']}")
    print("Conclusión:", regla["conclusion"])
print()

print("--- Caso 2: broca con infestación alta ---")
hechos_2 = {"plaga": "broca", "infestacion_pct": 12}
resultado_2 = motor.evaluate(hechos_2)
for regla in resultado_2["applied"]:
    print(f"[{regla['id']}] {regla['explanation']}")
    print("Conclusión:", regla["conclusion"])
print()

print("--- Caso 3: elegibilidad para asistencia técnica del MINAGRI ---")
hechos_3 = {"caficultor": True, "hectareas": 8}
resultado_3 = motor.evaluate(hechos_3)
for regla in resultado_3["applied"]:
    print(f"[{regla['id']}] {regla['explanation']}")
    print("Conclusión:", regla["conclusion"])

--- Caso 1: roya en época de lluvias ---
[roya-control-lluvias] Se cumple: (sintoma = pustulas_amarillas) Y (estacion = lluvias)
Conclusión: Aplicar fungicidas a base de cobre o triazoles y realizar podas fitosanitarias para reducir el inóculo.

--- Caso 2: broca con infestación alta ---
[broca-alta-infestacion] Se cumple: (plaga = broca) Y (infestacion_pct > 5)
Conclusión: Recolección completa y oportuna, manejo de frutos caídos y aplicación de Beauveria bassiana.

--- Caso 3: elegibilidad para asistencia técnica del MINAGRI ---
[elegibilidad-asistencia-tecnica] Se cumple: (caficultor es verdadero) Y (hectareas ≤ 10)
Conclusión: El productor es elegible para el programa de asistencia técnica gratuito (pequeño caficultor).


In [5]:
from ml_core.core.classifiers import ClassifierBundle

bundle = ClassifierBundle()
bundle.train(force=True)

print("--- Métricas de precisión (validación cruzada de 5 particiones) ---")
for resultado in bundle.metrics():
    print(f"{resultado['model']:15} accuracy={resultado['accuracy']:.3f} (±{resultado['accuracy_std']:.3f})  f1_macro={resultado['f1_macro']:.3f}")

--- Métricas de precisión (validación cruzada de 5 particiones) ---
decision_tree   accuracy=0.368 (±0.122)  f1_macro=0.308
naive_bayes     accuracy=0.496 (±0.120)  f1_macro=0.404
mlp             accuracy=0.432 (±0.039)  f1_macro=0.370


In [6]:
from ml_core.core.rl import QLearner

aprendiz = QLearner()

interacciones = [
    ("intent=roya|vision=alta|conocimiento=si", "knowledge_guided", 1),
    ("intent=roya|vision=alta|conocimiento=si", "llm_guided", -1),
    ("intent=roya|vision=alta|conocimiento=si", "knowledge_guided", 1),
    ("intent=broca|vision=baja|conocimiento=si", "classification_guided", 1),
    ("intent=broca|vision=baja|conocimiento=si", "knowledge_guided", -1),
]

print("--- Simulación de retroalimentación del usuario ---")
for estado, accion, recompensa in interacciones:
    accion_elegida = aprendiz.choose(estado)
    print(f"Estado: {estado}")
    print(f"  Acción que el sistema habría elegido ahora mismo: {accion_elegida}")
    print(f"  Acción evaluada por el usuario: {accion} -> recompensa {recompensa:+d}")
    aprendiz.update(estado, accion, recompensa)

print()
print("--- Tabla Q resultante tras la retroalimentación ---")
for estado, acciones in aprendiz.q.items():
    print(estado)
    for fuente, valor in acciones.items():
        print(f"  {fuente}: {valor:.4f}")

--- Simulación de retroalimentación del usuario ---
Estado: intent=roya|vision=alta|conocimiento=si
  Acción que el sistema habría elegido ahora mismo: knowledge_guided
  Acción evaluada por el usuario: knowledge_guided -> recompensa +1
Estado: intent=roya|vision=alta|conocimiento=si
  Acción que el sistema habría elegido ahora mismo: knowledge_guided
  Acción evaluada por el usuario: llm_guided -> recompensa -1
Estado: intent=roya|vision=alta|conocimiento=si
  Acción que el sistema habría elegido ahora mismo: knowledge_guided
  Acción evaluada por el usuario: knowledge_guided -> recompensa +1
Estado: intent=broca|vision=baja|conocimiento=si
  Acción que el sistema habría elegido ahora mismo: knowledge_guided
  Acción evaluada por el usuario: classification_guided -> recompensa +1
Estado: intent=broca|vision=baja|conocimiento=si
  Acción que el sistema habría elegido ahora mismo: classification_guided
  Acción evaluada por el usuario: knowledge_guided -> recompensa -1

--- Tabla Q resu

In [7]:
from ml_core.core.dialogue import update, discrepancy

print("--- Simulación de diálogo bayesiano ---")
print("Punto de partida: el modelo de visión detectó 'roya' con confianza moderada")
hipotesis_inicial = {"RUST": 0.5, "RED_SPIDER_MITE": 0.3, "HEALTHY": 0.2}
print("Hipótesis inicial:", hipotesis_inicial)
print()

print("Pregunta 1: ¿Qué observas principalmente en las hojas?")
print("Respuesta del usuario: 'Polvo anaranjado o amarillo'")
probs, top, confianza = update(hipotesis_inicial, "orange_powder", question_id="visual_symptom_confirmation")
print("Probabilidades actualizadas:", probs)
print(f"Hipótesis principal: {top} (confianza {confianza:.2%})")
print()

print("Pregunta 2: ¿Las manchas están debajo de la hoja?")
print("Respuesta del usuario: 'Sí, las manchas están debajo de la hoja'")
probs_2, top_2, confianza_2 = update(probs, "rust_under_leaf", question_id="rust_confirmation")
print("Probabilidades actualizadas:", probs_2)
print(f"Hipótesis principal: {top_2} (confianza {confianza_2:.2%})")
print()

diferencia = discrepancy(hipotesis_inicial, probs_2)
print(f"Discrepancia entre la hipótesis inicial y la final: {diferencia}")
if confianza_2 >= 0.75:
    print("Confianza suficiente (≥75%): el sistema concluiría el diagnóstico aquí.")
else:
    print("Confianza insuficiente: el sistema seguiría preguntando o pediría otra fotografía.")

--- Simulación de diálogo bayesiano ---
Punto de partida: el modelo de visión detectó 'roya' con confianza moderada
Hipótesis inicial: {'RUST': 0.5, 'RED_SPIDER_MITE': 0.3, 'HEALTHY': 0.2}

Pregunta 1: ¿Qué observas principalmente en las hojas?
Respuesta del usuario: 'Polvo anaranjado o amarillo'
Probabilidades actualizadas: {'RUST': 0.7458, 'RED_SPIDER_MITE': 0.2373, 'HEALTHY': 0.0169}
Hipótesis principal: RUST (confianza 74.58%)

Pregunta 2: ¿Las manchas están debajo de la hoja?
Respuesta del usuario: 'Sí, las manchas están debajo de la hoja'
Probabilidades actualizadas: {'RUST': 0.8062, 'RED_SPIDER_MITE': 0.1727, 'HEALTHY': 0.0211}
Hipótesis principal: RUST (confianza 80.62%)

Discrepancia entre la hipótesis inicial y la final: 0.3062
Confianza suficiente (≥75%): el sistema concluiría el diagnóstico aquí.


## Conclusión

Este notebook reproduce, usando el código real del repositorio del proyecto (no una reimplementación paralela), los cinco componentes exigidos por la rúbrica:

1. **Corpus e impacto**: 125 frases distribuidas en 20 intenciones sobre sanidad del cafeto, orientadas a pequeños caficultores dominicanos.
2. **Núcleo lógico**: búsqueda BFS, DFS y A* sobre el grafo de conocimiento, y motor de inferencia con 7 reglas de primer orden, incluyendo la regla de elegibilidad para el programa de asistencia técnica del MINAGRI.
3. **Interfaz cognitiva**: normalización de modismos dominicanos, tokenización con spaCy, y tres modelos de clasificación (árbol de decisión, Naive Bayes, perceptrón multicapa) evaluados con validación cruzada.
4. **Aprendizaje por refuerzo**: Q-learning real que ajusta sus preferencias de respuesta según la retroalimentación del usuario.
5. **Inferencia bayesiana**: actualización progresiva de hipótesis diagnósticas a partir de evidencia visual y textual, con umbral de confianza para decidir cuándo concluir.

Se observó que el algoritmo A* puede converger a un nodo distinto que BFS/DFS para la misma consulta, debido a que su heurística prioriza la similitud léxica general y no necesariamente el vocabulario técnico más preciso; esta observación queda documentada como hallazgo del proyecto. También se observó una ligera variación en las métricas de precisión de los clasificadores frente a las obtenidas en el entorno local, atribuible a diferencias de versión de scikit-learn entre Colab y el entorno de desarrollo.